# Heuristic Strategy Search v1

QuantBench-inspired factor/heuristic search for SOXL. This notebook screens simple causal position rules against the hard gate: every required regime window must have strategy max drawdown below 20% and strategy return above Buy&Hold.

External reference: arXiv 2504.18600 QuantBench emphasizes full-pipeline coverage, model/data standardization, distribution-shift robustness, and task-aligned portfolio metrics.

## Governance

- Direction: `model_selection`
- Role: diagnose / heuristic-search baseline
- Changed layer: strategy search and portfolio evaluation, not PPO training
- Baseline: existing replay hard-gate scan found zero passing profiles
- Acceptance: at least one config passes `test_2022` and `test_2023_2024` with MaxDD < 20% and return > Buy&Hold
- Failure mode: simple causal trend/risk rules remain too conservative in 2023-2024 or too exposed in 2022.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "SPEC.md").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "runtime/model_selection/card/heuristic_strategy_search_v1"
PANEL = ROOT / "data/processed/panels/soxl_soxx_vix_full.csv"
CARD.mkdir(parents=True, exist_ok=True)
print(ROOT)

/Users/rziiiii/.openclaw/workspace


In [2]:
import pandas as pd
from src.soxl_quant.app.heuristic_strategy_search import default_heuristic_config_grid, screen_heuristic_strategies

panel = pd.read_csv(PANEL)
configs = default_heuristic_config_grid()
rows = screen_heuristic_strategies(panel, configs, max_drawdown_pct=20.0)
rows.to_csv(CARD / "heuristic_strategy_screen.csv", index=False)
rows.head(12)

,config_id,gate_pass,score,momentum_lookback,soxx_momentum_lookback,require_soxx_confirmation,vix_threshold,bull_position,risk_position,cash_position,...,test_2022_max_drawdown_pct,test_2022_gate_pass,test_2022_failures,test_2023_2024_rows,test_2023_2024_strategy_return_pct,test_2023_2024_buyhold_return_pct,test_2023_2024_relative_return_pct,test_2023_2024_max_drawdown_pct,test_2023_2024_gate_pass,test_2023_2024_failures
0,236,False,493.104201,20,60,True,18.0,1.00,0.0,0.0,...,17.241801,True,,502,171.071669,191.773509,-20.701840,28.033556,False,"max_drawdown_ge_20,relative_return_not_positive"
1,239,False,500.799618,20,60,True,18.0,1.00,0.2,0.0,...,24.215522,False,max_drawdown_ge_20,502,173.231947,191.773509,-18.541563,31.799976,False,"max_drawdown_ge_20,relative_return_not_positive"
2,242,False,727.997243,20,60,True,18.0,1.00,0.4,0.0,...,30.891865,False,max_drawdown_ge_20,502,172.533312,191.773509,-19.240197,35.465424,False,"max_drawdown_ge_20,relative_return_not_positive"
3,230,False,2429.042643,20,60,True,18.0,0.85,0.2,0.0,...,21.951653,False,max_drawdown_ge_20,502,143.183837,191.773509,-48.589672,28.017323,False,"max_drawdown_ge_20,relative_return_not_positive"
4,227,False,2567.838010,20,60,True,18.0,0.85,0.0,0.0,...,14.769610,True,,502,141.261132,191.773509,-50.512377,24.042002,False,"max_drawdown_ge_20,relative_return_not_positive"
5,233,False,2640.971663,20,60,True,18.0,0.85,0.4,0.0,...,28.827435,False,max_drawdown_ge_20,502,142.562034,191.773509,-49.211476,31.886071,False,"max_drawdown_ge_20,relative_return_not_positive"
6,290,False,2907.489809,20,60,True,26.0,1.00,0.0,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"
7,293,False,2907.489809,20,60,True,26.0,1.00,0.2,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"
8,296,False,2907.489809,20,60,True,26.0,1.00,0.4,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"
9,317,False,2907.489809,20,60,True,32.0,1.00,0.0,0.0,...,48.993258,False,max_drawdown_ge_20,502,154.380950,191.773509,-37.392559,45.858796,False,"max_drawdown_ge_20,relative_return_not_positive"


In [3]:
passing = rows.loc[rows["gate_pass"] == True].copy()
summary = {
    "experiment": "heuristic_strategy_search_v1",
    "decision": "hard_gate_pass_found" if len(passing) else "hard_gate_no_heuristic_passes",
    "promotion_allowed": bool(len(passing)),
    "training_allowed_from_this_card": False,
    "configs": int(len(rows)),
    "passing_configs": int(len(passing)),
    "max_drawdown_pct": 20.0,
    "required_windows": ["test_2022", "test_2023_2024"],
    "best": rows.head(10).to_dict(orient="records"),
    "next_action": "Add a stronger reentry/ensemble or rolling learner branch; simple causal momentum/VIX heuristics did not clear the 2023-2024 return plus drawdown conflict.",
    "sidecars": ["summary.json", "heuristic_strategy_screen.csv"],
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
summary

{'experiment': 'heuristic_strategy_search_v1',
 'decision': 'hard_gate_no_heuristic_passes',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'configs': 2592,
 'passing_configs': 0,
 'max_drawdown_pct': 20.0,
 'required_windows': ['test_2022', 'test_2023_2024'],
 'best': [{'config_id': 236,
   'gate_pass': False,
   'score': 493.1042013907361,
   'momentum_lookback': 20,
   'soxx_momentum_lookback': 60,
   'require_soxx_confirmation': True,
   'vix_threshold': 18.0,
   'bull_position': 1.0,
   'risk_position': 0.0,
   'cash_position': 0.0,
   'drawdown_start': 0.99,
   'drawdown_full': 1.0,
   'drawdown_min_position': 0.0,
   'test_2022_rows': 251,
   'test_2022_strategy_return_pct': -17.241801,
   'test_2022_buyhold_return_pct': -86.588072,
   'test_2022_relative_return_pct': 69.346271,
   'test_2022_max_drawdown_pct': 17.241801,
   'test_2022_gate_pass': True,
   'test_2022_failures': '',
   'test_2023_2024_rows': 502,
   'test_2023_2024_strategy_return_pct': 

## Conclusion

The v1 heuristic-search layer is now reusable and fast enough for repeated candidate scans. On the current SOXL daily panel, the default 2,592-rule grid found **zero** configs passing both required windows. The best candidate cleared 2022 with `-17.24%` strategy return vs `-86.59%` Buy&Hold and `17.24%` MaxDD, but failed 2023-2024 with `171.07%` strategy return vs `191.77%` Buy&Hold and `28.03%` MaxDD.

Decision: do not promote this simple heuristic family. Next action is to add a stronger reentry/ensemble or rolling learner branch that improves 2023-2024 participation without giving back the 2022 drawdown control.